## Tapis and MINT Pipeline
#### 1. Dependencies Installation and Setup Secrets

In [10]:
# Installation of the other standard libraries
%pip install python-dotenv tapipy urllib3

Note: you may need to restart the kernel to use updated packages.


In [13]:
import os
import json
import base64
import urllib3
import requests
from dotenv import load_dotenv
from tapipy.tapis import Tapis

load_dotenv()
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class MintMinimalClient:
    def __init__(self, base_url, user):
        # MINT suele requerir el usuario en la URL para el Model Catalog
        self.base_url = base_url.rstrip('/')
        self.user = user

    def post_resource(self, resource_type, data):
        # Intentamos el formato: base_url/user/resource_type
        # Ejemplo: https://api.models.mint.isi.edu/v1/s_sterling22/models
        url = f"{self.base_url}/{self.user}/{resource_type}"
        
        print(f"DEBUG: Intentando POST a {url}")
        response = requests.post(url, json=data)
        
        if response.status_code in [200, 201]:
            return response.json()
        elif response.status_code == 404:
            # Fallback: Intentar sin el usuario en la URL si el anterior falla
            url_fallback = f"{self.base_url}/{resource_type}"
            print(f"DEBUG: 404 detectado. Intentando Fallback a {url_fallback}")
            response = requests.post(url_fallback, json=data)
            if response.status_code in [200, 201]:
                return response.json()
        
        raise Exception(f"Error en MINT ({response.status_code}): {response.text}")

# Mint Initialization
MINT_API_URL = "https://api.models.mint.isi.edu/v1"
# Asegúrate de que este usuario coincida con el que usas en el portal de MINT
MINT_USER = os.getenv('TAPIS_USERNAME') or "s_sterling22" 

mint = MintMinimalClient(MINT_API_URL, MINT_USER)

# Tapis Client
t = Tapis(
    base_url=os.getenv('TAPIS_BASE_URL'),
    username=os.getenv('TAPIS_USERNAME'),
    password=os.getenv('TAPIS_PASSWORD'),
    verify=False
)
t.get_tokens()

print(f"✅ Clients ready. Using Direct API for MINT.")

✅ Clients ready. Using Direct API for MINT.


#### 2. Client Initialization (Tapis & MINT)

In [15]:
try:
    # 1. Crear Modelo
    model_data = {
        "type": ["Model"], # Especificar el tipo para la ontología
        "label": ["Tapis ETL Framework"],
        "description": ["Framework modular para orquestar pipelines de Tapis v3."],
        "hasModelCategory": ["Data Science"]
    }
    model_res = mint.post_resource("models", model_data)
    model_id = model_res['id']
    print(f"✅ Modelo MINT Creado: {model_id}")

    # 2. Crear Software Version
    software_data = {
        "type": ["SoftwareVersion"],
        "label": ["v1.0.0-tapis"],
        "hasSoftwareImage": ["ghcr.io/sethsterling22/tapis-workflow-automation/tapis-dummy-app:latest"]
    }
    soft_res = mint.post_resource("softwareversions", software_data)
    soft_id = soft_res['id']
    print(f"✅ Software Version Creada: {soft_id}")

    # 3. Crear Configuración
    tapis_metadata = {
        "pipeline_id": "hello-world-pipeline",
        "group_id": os.getenv('TAPIS_GROUP_ID'),
        "base_url": os.getenv('TAPIS_BASE_URL')
    }
    
    config_data = {
        "type": ["ModelConfiguration"],
        "label": ["Tapis Pipeline Config"],
        "description": [json.dumps(tapis_metadata)],
        "hasSoftwareImage": ["ghcr.io/sethsterling22/tapis-workflow-automation/tapis-dummy-app:latest"]
    }
    config_res = mint.post_resource("modelconfigurations", config_data)
    CONFIG_ID = config_res['id']
    
    print(f"✅ CONFIGURACIÓN FINAL: {CONFIG_ID}")

except Exception as e:
    print(f"❌ {e}")

DEBUG: Intentando POST a https://api.models.mint.isi.edu/v1/s_sterling22/models
DEBUG: 404 detectado. Intentando Fallback a https://api.models.mint.isi.edu/v1/models
❌ Error en MINT (404): {"detail":"Not Found"}


In [ ]:
import time

def trigger_tapis_from_mint(config_url):
    # En un entorno real, MINT pasaría el ID de la configuración seleccionada
    print(f"🔍 Recuperando metadatos de la configuración MINT...")
    
    # Obtener la descripción que contiene los datos de Tapis
    resp = requests.get(config_url)
    config_data = resp.json()
    tapis_info = json.loads(config_data['description'][0])
    
    p_id = tapis_info['pipeline_id']
    g_id = tapis_info['group_id']

    print(f"🚀 Disparando Tapis Pipeline: {p_id}")
    run = t.workflows.runPipeline(group_id=g_id, pipeline_id=p_id)
    run_uuid = run.uuid

    # Loop de monitoreo
    while True:
        status_info = t.workflows.getPipelineRun(group_id=g_id, pipeline_id=p_id, pipeline_run_uuid=run_uuid)
        print(f"[{time.strftime('%H:%M:%S')}] Status: {status_info.status}")
        if status_info.status == 'completed':
            print("✅ Tarea completada. Pipeline ETL listo para el siguiente paso.")
            break
        elif status_info.status in ['failed', 'cancelled']:
            print("❌ El pipeline falló.")
            break
        time.sleep(10)

# Ejecutar (usando el ID obtenido en la celda anterior)
# trigger_tapis_from_mint(CONFIG_ID)

#### 1. EndPoint Configuration

In [16]:
import os
import requests
import json

DCAT_URL = "https://api.data.mint.isi.edu/v1" # Ajustar según tu entorno (ISI o local)
PROVENANCE_ID = "tapis-workflow-service" # Identificador para tus pipelines

print(f"✅ Configurado para enviar datos a: {DCAT_URL}")

✅ Configurado para enviar datos a: https://api.data.mint.isi.edu/v1


#### 2. Registration of Tapis Output as a MINT Dataset

In [ ]:
def register_tapis_output_in_mint(pipeline_run_uuid, dataset_name):
    # 1. Definir el Dataset (Metadata)
    payload = {
        "datasets": [{
            "provenance_id": PROVENANCE_ID,
            "name": dataset_name,
            "description": f"Output generado por Tapis Run {pipeline_run_uuid}",
            "metadata": {"source": "Tapis-Workflows", "run_id": pipeline_run_uuid}
        }]
    }
    
    # 2. Llamada al DCAT (siguiendo la lógica de tu jefe)
    r = requests.post(f"{DCAT_URL}/datasets/register_datasets", json=payload)
    if r.status_code == 200:
        dsid = r.json()["datasets"][0]["record_id"]
        print(f"✅ Dataset registrado en MINT Data Catalog: {dsid}")
        return dsid
    else:
        print(f"❌ Error en DCAT: {r.text}")
        return None

#### 3. Resource registry

In [ ]:
def register_resource_to_dataset(dsid, file_url):
    resource_payload = {
        "resources": [{
            "provenance_id": PROVENANCE_ID,
            "dataset_id": dsid,
            "name": os.path.basename(file_url),
            "url": file_url, # URL de descarga desde Tapis
            "metadata": {"format": "csv"}
        }]
    }
    r = requests.post(f"{DCAT_URL}/datasets/register_resources", json=resource_payload)
    return r.status_code == 200